In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,ConfusionMatrixDisplay,classification_report
np.random.seed(9); tf.random.set_seed(9)
# Reproducible telecom customer-churn dataset
n=1200
df=pd.DataFrame({"tenure":np.random.randint(1,73,n),"monthly_charges":np.random.uniform(25,120,n),"complaints":np.random.poisson(1.2,n),"data_usage":np.random.uniform(1,100,n),"contract_type":np.random.choice(["Month-to-month","One-year","Two-year"],n,p=[.55,.25,.20])})
logit=-1.0-.035*df.tenure+.018*df.monthly_charges+.48*df.complaints+1.0*(df.contract_type=="Month-to-month")-.5*(df.contract_type=="Two-year")
prob=1/(1+np.exp(-logit)); df["Churn"]=np.random.binomial(1,prob)
X=pd.get_dummies(df.drop(columns="Churn"),columns=["contract_type"],dtype=float); y=df.Churn
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=9,stratify=y)
sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)
def make_mlp(width):
 m=tf.keras.Sequential([tf.keras.layers.Input((Xtr.shape[1],)),tf.keras.layers.Dense(width,activation="relu"),tf.keras.layers.Dense(max(8,width//2),activation="relu"),tf.keras.layers.Dense(1,activation="sigmoid")])
 m.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"]); return m
model=make_mlp(32); hist=model.fit(Xtr,ytr,validation_split=.2,epochs=30,batch_size=32,verbose=1)
p=model.predict(Xte,verbose=0).ravel(); pred=(p>=.5).astype(int)
print("Accuracy:",round(accuracy_score(yte,pred),4),"Precision:",round(precision_score(yte,pred,zero_division=0),4),"Recall:",round(recall_score(yte,pred,zero_division=0),4),"F1:",round(f1_score(yte,pred,zero_division=0),4))
print(classification_report(yte,pred,zero_division=0)); print("Confusion matrix:\n",confusion_matrix(yte,pred))
fig,ax=plt.subplots(1,2,figsize=(11,4)); ax[0].plot(hist.history["accuracy"],label="Train"); ax[0].plot(hist.history["val_accuracy"],label="Validation"); ax[0].set_title("Accuracy"); ax[1].plot(hist.history["loss"],label="Train"); ax[1].plot(hist.history["val_loss"],label="Validation"); ax[1].set_title("Loss")
for a in ax:a.legend();a.grid()
plt.tight_layout();plt.show()
ConfusionMatrixDisplay.from_predictions(yte,pred);plt.tight_layout();plt.show()
width_results=[]
for width in [8,16,32,64]:
 m=make_mlp(width); m.fit(Xtr,ytr,epochs=15,batch_size=32,verbose=0); q=(m.predict(Xte,verbose=0).ravel()>=.5).astype(int); width_results.append([width,accuracy_score(yte,q)])
print(pd.DataFrame(width_results,columns=["Hidden width","Test accuracy"]).to_string(index=False))
plt.plot([r[0] for r in width_results],[r[1] for r in width_results],marker="o");plt.xlabel("Hidden-layer width");plt.ylabel("Test accuracy");plt.title("Effect of hidden-layer width");plt.grid();plt.show()